# 파이썬 심화 문제집 - 클래스

## 클래스 심화 (471 ~ 480)

### 471 클래스 - 간단한 HTTP 클라이언트 시뮬레이션

HTTP 요청/응답을 시뮬레이션하는 클래스를 구현하세요.

<details>
<summary>정답 확인 (클릭!)</summary>
<pre>
class Response:
    def __init__(self,status,body): self.status=status; self.body=body
    def __repr__(self): return f'Response({self.status}, {self.body})'

class MockHTTP:
    def __init__(self):
        self._routes={'GET':{},'POST':{}}
    def register(self,method,path,response):
        self._routes[method][path]=response
    def request(self,method,path,data=None):
        if path in self._routes.get(method,{}):
            return self._routes[method][path]
        return Response(404,'Not Found')

client=MockHTTP()
client.register('GET','/users',Response(200,{'users':['김파이','이파이']}))
client.register('POST','/users',Response(201,'사용자 생성됨'))
print(client.request('GET','/users'))
print(client.request('POST','/users'))
print(client.request('GET','/unknown'))
</pre>
</details>

In [ ]:
# 여기에 코드를 작성하세요


### 472 클래스 - 불변 클래스 (Immutable)

생성 후 속성 변경이 불가능한 불변 클래스를 구현하세요.

<details>
<summary>정답 확인 (클릭!)</summary>
<pre>
class Point:
    def __init__(self,x,y):
        object.__setattr__(self,'x',x)
        object.__setattr__(self,'y',y)
    def __setattr__(self,name,value):
        raise AttributeError('Point는 불변 객체입니다')
    def __repr__(self): return f'Point({self.x},{self.y})'
    def translate(self,dx,dy): return Point(self.x+dx,self.y+dy)

p=Point(1,2)
print(p)
p2=p.translate(3,4)
print(p2)
try:
    p.x=10  # 오류
except AttributeError as e:
    print(e)
</pre>
</details>

In [ ]:
# 여기에 코드를 작성하세요


### 473 클래스 - 슬롯 최적화 (__slots__)

__slots__를 사용해서 메모리를 최적화하는 클래스를 만들고 일반 클래스와 비교하세요.

<details>
<summary>정답 확인 (클릭!)</summary>
<pre>
import sys

class NormalPoint:
    def __init__(self,x,y): self.x=x; self.y=y

class SlottedPoint:
    __slots__=['x','y']
    def __init__(self,x,y): self.x=x; self.y=y

p1=NormalPoint(1,2)
p2=SlottedPoint(1,2)
print('일반 클래스 크기:',sys.getsizeof(p1.__dict__),'bytes')
print('슬롯 클래스: __dict__ 없음')
print('속성 접근:', p1.x, p2.x)
</pre>
</details>

In [ ]:
# 여기에 코드를 작성하세요


### 474 클래스 - 제너레이터 기반 파이프라인

제너레이터를 활용한 데이터 처리 파이프라인을 클래스로 구현하세요.

<details>
<summary>정답 확인 (클릭!)</summary>
<pre>
class DataPipeline:
    def __init__(self,data): self.gen=iter(data)
    def filter(self,pred):
        self.gen=(x for x in self.gen if pred(x))
        return self
    def map(self,func):
        self.gen=(func(x) for x in self.gen)
        return self
    def take(self,n):
        result=[]
        for _ in range(n):
            try: result.append(next(self.gen))
            except StopIteration: break
        return result
    def collect(self): return list(self.gen)

result=(DataPipeline(range(1,100))
    .filter(lambda x:x%3==0)
    .map(lambda x:x**2)
    .take(5))
print(result)
</pre>
</details>

In [ ]:
# 여기에 코드를 작성하세요


### 475 클래스 - 메타클래스 기초

메타클래스를 사용해서 모든 메서드 호출을 로깅하는 클래스를 만드세요.

<details>
<summary>정답 확인 (클릭!)</summary>
<pre>
class LogMeta(type):
    def __new__(mcs,name,bases,attrs):
        for attr_name,attr_val in attrs.items():
            if callable(attr_val) and not attr_name.startswith('_'):
                attrs[attr_name]=mcs._wrap(attr_name,attr_val)
        return super().__new__(mcs,name,bases,attrs)
    @staticmethod
    def _wrap(name,func):
        def wrapper(*args,**kwargs):
            print(f'호출: {name}')
            return func(*args,**kwargs)
        return wrapper

class Calculator(metaclass=LogMeta):
    def add(self,a,b): return a+b
    def mul(self,a,b): return a*b

c=Calculator()
print(c.add(3,4))
print(c.mul(3,4))
</pre>
</details>

In [ ]:
# 여기에 코드를 작성하세요


### 476 클래스 - 커링 (Currying)

커링을 지원하는 함수 클래스를 구현하세요.

<details>
<summary>정답 확인 (클릭!)</summary>
<pre>
class Curry:
    def __init__(self,func): self.func=func; self.args=[]
    def __call__(self,*args):
        import inspect
        self.args=list(args)
        n=len(inspect.signature(self.func).parameters)
        if len(self.args)>=n:
            return self.func(*self.args[:n])
        return self
    def __repr__(self): return f'Curry({self.func.__name__}, args={self.args})'

@Curry
def add(a,b,c): return a+b+c

print(add(1)(2)(3))  # 6
print(add(1,2,3))   # 6
</pre>
</details>

In [ ]:
# 여기에 코드를 작성하세요


### 477 클래스 - 리소스 풀

연결 풀처럼 리소스를 재사용하는 ResourcePool 클래스를 구현하세요.

<details>
<summary>정답 확인 (클릭!)</summary>
<pre>
class Resource:
    _id_counter=0
    def __init__(self):
        Resource._id_counter+=1
        self.id=Resource._id_counter
        self.in_use=False
    def __repr__(self): return f'Resource({self.id})'

class ResourcePool:
    def __init__(self,size): self.pool=[Resource() for _ in range(size)]
    def acquire(self):
        for r in self.pool:
            if not r.in_use: r.in_use=True; return r
        return None
    def release(self,r): r.in_use=False
    def status(self): print([f'{r}:{"사용중" if r.in_use else "가용"}' for r in self.pool])

pool=ResourcePool(3)
pool.status()
r1=pool.acquire(); r2=pool.acquire()
pool.status()
pool.release(r1)
pool.status()
</pre>
</details>

In [ ]:
# 여기에 코드를 작성하세요


### 478 클래스 - 플러그인 시스템

플러그인을 동적으로 등록하고 실행하는 시스템을 구현하세요.

<details>
<summary>정답 확인 (클릭!)</summary>
<pre>
class PluginManager:
    _plugins={}
    @classmethod
    def register(cls,name):
        def decorator(plugin_cls): cls._plugins[name]=plugin_cls; return plugin_cls
        return decorator
    @classmethod
    def get(cls,name): return cls._plugins.get(name)
    @classmethod
    def list_plugins(cls): return list(cls._plugins.keys())

@PluginManager.register('json')
class JSONFormatter:
    def format(self,data):
        import json; return json.dumps(data,ensure_ascii=False)

@PluginManager.register('csv')
class CSVFormatter:
    def format(self,data): return ','.join(str(v) for v in data.values())

data={'name':'김파이','age':25,'city':'서울'}
for name in PluginManager.list_plugins():
    fmt=PluginManager.get(name)()
    print(f'{name}:', fmt.format(data))
</pre>
</details>

In [ ]:
# 여기에 코드를 작성하세요


### 479 클래스 - 간단한 규칙 엔진

조건과 액션을 등록하고 실행하는 규칙 엔진을 구현하세요.

<details>
<summary>정답 확인 (클릭!)</summary>
<pre>
class Rule:
    def __init__(self,name,condition,action):
        self.name=name; self.condition=condition; self.action=action
    def apply(self,data):
        if self.condition(data): self.action(data); return True
        return False

class RuleEngine:
    def __init__(self): self.rules=[]
    def add(self,rule): self.rules.append(rule)
    def run(self,data):
        for rule in self.rules:
            if rule.apply(data): print(f'규칙 [{rule.name}] 적용됨')

engine=RuleEngine()
engine.add(Rule('고가상품', lambda d:d['price']>100000, lambda d: d.update({'category':'premium'})))
engine.add(Rule('재고부족', lambda d:d['stock']<10, lambda d: d.update({'alert':'재고부족경고'})))

product={'name':'노트북','price':1500000,'stock':3}
engine.run(product)
print(product)
</pre>
</details>

In [ ]:
# 여기에 코드를 작성하세요


### 480 클래스 종합 - 미니 SQL 엔진

간단한 SQL SELECT 쿼리를 처리하는 미니 엔진을 구현하세요.

<details>
<summary>정답 확인 (클릭!)</summary>
<pre>
class Table:
    def __init__(self,name,columns):
        self.name=name; self.columns=columns; self.rows=[]
    def insert(self,*values):
        self.rows.append(dict(zip(self.columns,values)))
    def select(self,cols='*',where=None,order=None,limit=None):
        result=[r for r in self.rows if (not where or where(r))]
        if order: result.sort(key=lambda r:r[order])
        if limit: result=result[:limit]
        if cols=='*': return result
        return [{c:r[c] for c in cols} for r in result]

users=Table('users',['id','name','age','city'])
for row in [(1,'김파이',25,'서울'),(2,'이파이',30,'부산'),(3,'박파이',22,'서울'),(4,'최파이',35,'대구')]:
    users.insert(*row)
print('서울 사용자:', users.select(where=lambda r:r['city']=='서울'))
print('나이순:', users.select(cols=['name','age'],order='age',limit=3))
</pre>
</details>

In [ ]:
# 여기에 코드를 작성하세요
